<a href="https://colab.research.google.com/github/Samir-atra/Barbados_Traffic_Analysis_Challenge_dev/blob/main/notebooks/keras_classifier_lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LSTM classifier with keras and polars

In [1]:
!pip install --upgrade decord

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 100.4 MB/s eta 0:00:0000:01:01


## Imports

In [2]:
# Imports

import os
import cv2
import keras
import subprocess
import numpy as np
import pandas as pd
import polars as pl
import tensorflow as tf
import matplotlib.pyplot as plt
from google.cloud import storage
from collections import defaultdict
from IPython.display import Video, display
from google.colab.patches import cv2_imshow

2026-01-09 17:27:32.339965: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767979652.538310      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767979652.594476      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767979653.060102      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767979653.060146      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767979653.060151      55 computation_placer.cc:177] computation placer alr

## Mount google drive

In [3]:
# from google.colab import drive
# drive.mount('/content/drive')

## Authenticate the cloud login


In [4]:
import os
from google.oauth2 import service_account

# If you uploaded the file to a dataset:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/kaggle/input/normanniles-traffic-authentication/nifty-structure-482811-m3-f328999c18d2.json"

# os.system("gcloud auth login")  # Authenticates your identity for general gcloud CLI command use e.g. gcloud storage cp, gcloud compute, gsuti
# os.system("gcloud auth application-default login")

In [5]:
client = storage.Client(project="brb-traffic")

# Base directories and bucket name
base_dir = "/kaggle/working/"
bucket_name = 'brb-traffic'

# Video paths
video_dir = os.path.join(base_dir, 'videos')
video_path = 'videos'
os.makedirs(video_dir, exist_ok=True)

# Datasheet paths
train_csv_path = os.path.join(base_dir, 'Train.csv')
sample_submission_csv_path = os.path.join(base_dir, 'SampleSubmission.csv')

## Read data

In [6]:
# Load the dataset
train = pd.read_csv(os.path.join(base_dir, '/kaggle/input/barbados-traffic-analysis-challenge/Train.csv'))

# Extract camera ID and reconstruct the video path
def extract_camera_path(video):
    parts = video.split('/')
    filename = parts[-1]
    camera_id = filename.split('_')[0]
    return f"{camera_id}/{filename}"

train['videos'] = train['videos'].apply(extract_camera_path)

# Display shape and preview
display(train.shape, train.head())

(16076, 14)

,responseId,view_label,ID_enter,ID_exit,videos,video_time,datetimestamp_start,datetimestamp_end,date,signaling,congestion_enter_rating,congestion_exit_rating,time_segment_id,cycle_phase
0,zYkHaeOdB7XOnvgP3YW5kQs,Norman Niles #1,time_segment_0_Norman Niles #1_congestion_ente...,time_segment_0_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-00-45.mp4,2025-10-20 06:00:45,2025-10-20 06:00:45,2025-10-20 06:01:44,2025-10-20,none,free flowing,free flowing,0,train
1,NYsHaeCRLq-vnvgPjoXZqA0,Norman Niles #1,time_segment_1_Norman Niles #1_congestion_ente...,time_segment_1_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-01-45.mp4,2025-10-20 06:01:45,2025-10-20 06:01:45,2025-10-20 06:02:44,2025-10-20,none,free flowing,free flowing,1,train
2,A40HaYT8KNm7nvgPq8e12AU,Norman Niles #1,time_segment_2_Norman Niles #1_congestion_ente...,time_segment_2_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-02-45.mp4,2025-10-20 06:02:45,2025-10-20 06:02:45,2025-10-20 06:03:00,2025-10-20,none,free flowing,free flowing,2,train
3,EIsHaanDMK-vnvgPjoXZqA0,Norman Niles #1,time_segment_3_Norman Niles #1_congestion_ente...,time_segment_3_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-03-45.mp4,2025-10-20 06:03:45,2025-10-20 06:03:45,2025-10-20 06:04:44,2025-10-20,none,free flowing,free flowing,3,train
4,RYsHafSeMaqpmecP5vCV0AQ,Norman Niles #1,time_segment_4_Norman Niles #1_congestion_ente...,time_segment_4_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-04-45.mp4,2025-10-20 06:04:45,2025-10-20 06:04:45,2025-10-20 06:04:59,2025-10-20,none,free flowing,free flowing,4,train


## Read sample submission

In [7]:
ss = pd.read_csv(os.path.join(base_dir,'/kaggle/input/barbados-traffic-analysis-challenge/SampleSubmission.csv'))
display(ss.shape,ss.head())

(880, 3)

,ID,Target,Target_Accuracy
0,time_segment_129_Norman Niles #1_congestion_en...,free flowing,free flowing
1,time_segment_130_Norman Niles #1_congestion_en...,heavy delay,heavy delay
2,time_segment_131_Norman Niles #1_congestion_en...,free flowing,free flowing
3,time_segment_132_Norman Niles #1_congestion_en...,heavy delay,heavy delay
4,time_segment_133_Norman Niles #1_congestion_en...,free flowing,free flowing


## Download videos

In [8]:
%%capture
blobs=train.videos.tolist()[0:2000]
print(f"Number of blobs selected: {len(blobs)}")
display(blobs)

In [9]:
%%capture
from google.api_core.exceptions import NotFound

print(f"--- Debugging Blobs Variable ---")
print(f"Type of 'blobs' before loop: {type(blobs)}")
print(f"Content of 'blobs' (first 5): {blobs[:5]}")
print(f"----------------------------------")

# --- Existing loop for downloading files from the 'blobs' list ---
for blob_name in blobs:
    # Ensure blob_name is a string before proceeding
    if not isinstance(blob_name, str):
        print(f"❌ Error: Expected string for blob_name, but got {type(blob_name)}. Skipping.")
        continue

    blob = client.bucket(bucket_name).blob(blob_name)
    file_name = os.path.basename(blob_name)  # get last part after '/'
    local_path = os.path.join(video_path, file_name)

    print(f"Attempting to download blob: '{blob_name}' to '{local_path}'") # Added print for clarity
    try:
        blob.download_to_filename(local_path)
        print("✅ Downloaded to:", local_path)
    except NotFound:
        print(f"❌ Error: '{blob_name}' not found in bucket '{bucket_name}'. Skipping.")
    except Exception as e:
        print(f"❌ An unexpected error occurred while downloading '{blob_name}': {e}")

## Create the data loading and processing functions

In [10]:
###### to be used but possibily to jax ######
import os
import cv2
import keras
import subprocess
import numpy as np
import pandas as pd
import polars as pl
import tensorflow as tf
import matplotlib.pyplot as plt
from google.cloud import storage
from collections import defaultdict
from IPython.display import Video, display
from google.colab.patches import cv2_imshow
import decord # Import decord
from decord import VideoReader, cpu # Import VideoReader and cpu for efficient frame loading

def load_and_preprocess_video(video_path, target_size=(224, 224), max_frames=None):
    """
    Loads a video, extracts frames using decord, resizes them, converts to RGB, and normalizes pixel values.

    Args:
        video_path (str): The path to the video file.
        target_size (tuple): A tuple (width, height) specifying the desired frame size.
        max_frames (int, optional): The maximum number of frames to extract. If None, all frames are extracted.

    Returns:
        list: A list of preprocessed frames (numpy arrays) or an empty list if the video cannot be opened.
    """
    frames = []
    try:
        vr = VideoReader(video_path, ctx=cpu(0)) # Use CPU context for VideoReader
    except Exception as e:
        print(f"Error: Could not open video {video_path} with decord: {e}")
        return []

    total_frames = len(vr)
    indices = np.arange(0, total_frames)
    if max_frames is not None and max_frames < total_frames:
        # If max_frames is less than total_frames, sample evenly distributed frames
        step = total_frames // max_frames
        if step == 0: # Handle cases where max_frames is close to total_frames
             indices = np.arange(0, total_frames)
        else:
            indices = np.arange(0, total_frames, step)[:max_frames]
    else:
        # Use all frames if max_frames is None or larger than total_frames
        indices = np.arange(0, total_frames)

    # Only decode the selected frames
    vr.seek(0) # Ensure we start from the beginning for indexing
    batch_frames = vr.get_batch(indices).asnumpy() # Decode frames as numpy array

    for frame in batch_frames:
        # decord typically returns frames in RGB format already
        # No need for cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Resize frame using OpenCV
        resized_frame = cv2.resize(frame, target_size, interpolation=cv2.INTER_AREA)

        # Normalize pixel values to [0, 1]
        normalized_frame = resized_frame.astype(np.float32) / 255.0

        frames.append(normalized_frame)

    # print(f"Processed {len(frames)} frames from {video_path}") # Commented out to reduce verbose output
    return frames

## generate the dataset

In [11]:
##### To be used but possibily to jax #######
import math
import tensorflow as tf
import numpy as np # Retain numpy import as it's used in this cell
import pandas as pd # Added for train_df_local
import os # Added for scanning video directory
from sklearn.model_selection import train_test_split # Added for X_train, y_train
import re # Import regex for string parsing

import cv2 # Import cv2 for image processing
import decord # Import decord
from decord import VideoReader, cpu, gpu # Import VideoReader and cpu/gpu for efficient frame loading

# --- NEW: Configuration variable for Decord device ---
DECODORD_USE_GPU = False # Set to True to use GPU, False to use CPU
# -----------------------------------------------------

# Constant for the four delay states
ALL_FOUR_DELAY_STATES = ['free_flowing', 'low_delay', 'moderate_delay', 'heavy_delay']

# Load the dataset
train = pd.read_csv(os.path.join('/kaggle/input/barbados-traffic-analysis-challenge', 'Train.csv'))

# Extract camera ID and reconstruct the video path
def extract_camera_path(video):
    parts = video.split('/')
    filename = parts[-1]
    camera_id = filename.split('_')[0]
    return f"{camera_id}/{filename}"

train['videos'] = train['videos'].apply(extract_camera_path)

# Helper function to extract delay category from ID string
# This function is no longer directly used for ID_enter/ID_exit in train_df_local creation
def extract_delay_category_from_id(id_string):
    match = re.search(r'congestion_(entrance|exit)_(free_flowing|low_delay|moderate_delay|heavy_delay)', id_string)
    if match:
        return match.group(2) # Return 'free_flowing', 'low_delay', etc.
    return None

def load_and_preprocess_video(video_path, target_size=(224, 224), max_frames=None):
    """
    Loads a video, extracts frames using decord, resizes them, converts to RGB, and normalizes pixel values.

    Args:
        video_path (str): The path to the video file.
        target_size (tuple): A tuple (width, height) specifying the desired frame size.
        max_frames (int, optional): The maximum number of frames to extract. If None, all frames are extracted.

    Returns:
        list: A list of preprocessed frames (numpy arrays) or an empty list if the video cannot be opened.
    """
    frames = []
    try:
        # --- UPDATED: Use DECODORD_USE_GPU to select context ---
        if DECODORD_USE_GPU:
            vr = VideoReader(video_path, ctx=gpu(0)) # Use GPU context
        else:
            vr = VideoReader(video_path, ctx=cpu(0))  # Use CPU context
        # -------------------------------------------------------
    except Exception as e:
        print(f"Error: Could not open video {video_path} with decord: {e}")
        return []

    total_frames = len(vr)
    indices = np.arange(0, total_frames)
    if max_frames is not None and max_frames < total_frames:
        # If max_frames is less than total_frames, sample evenly distributed frames
        step = total_frames // max_frames
        if step == 0: # Handle cases where max_frames is close to total_frames
             indices = np.arange(0, total_frames)
        else:
            indices = np.arange(0, total_frames, step)[:max_frames]
    else:
        # Use all frames if max_frames is None or larger than total_frames
        indices = np.arange(0, total_frames)

    # Only decode the selected frames
    vr.seek(0) # Ensure we start from the beginning for indexing
    batch_frames = vr.get_batch(indices).asnumpy() # Decode frames as numpy array

    for frame in batch_frames:
        # decord typically returns frames in RGB format already
        # No need for cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Resize frame using OpenCV
        resized_frame = cv2.resize(frame, target_size, interpolation=cv2.INTER_AREA)

        # Normalize pixel values to [0, 1]
        normalized_frame = resized_frame.astype(np.float32) / 255.0

        frames.append(normalized_frame)

    # print(f"Processed {len(frames)} frames from {video_path}") # Commented out to reduce verbose output
    return frames

# Re-create train_df_local based on video files in /content/videos
video_files = [f for f in os.listdir('/kaggle/working/videos') if f.endswith(('.mp4', '.avi', '.mov', '.mkv'))]
local_video_paths = [os.path.join('/kaggle/working/videos', f) for f in video_files]

# Create a mapping from filename in /content/videos to the full path in `train['videos']`
filename_to_full_path = {os.path.basename(p): p for p in train['videos'].tolist()}

# Prepare a list to hold data for the new train_df_local, now including the dominant_combined_label
data_for_local_df = []

for local_path in local_video_paths:
    filename = os.path.basename(local_path)
    # Find the corresponding entry in the original `train` DataFrame using the processed 'videos' column
    original_video_path_in_train_df = filename_to_full_path.get(filename)

    if original_video_path_in_train_df:
        # Get the row from the `train` DataFrame that matches this video
        matching_row = train[train['videos'] == original_video_path_in_train_df].iloc[0]

        # --- FIXED: Extract delay categories from 'congestion_enter_rating' and 'congestion_exit_rating' columns ---
        entrance_delay_str = matching_row['congestion_enter_rating']
        exit_delay_str = matching_row['congestion_exit_rating']

        # Normalize strings: replace spaces with underscores to match ALL_FOUR_DELAY_STATES
        if isinstance(entrance_delay_str, str):
            entrance_delay_str = entrance_delay_str.replace(' ', '_')
        if isinstance(exit_delay_str, str):
            exit_delay_str = exit_delay_str.replace(' ', '_')
        # ----------------------------------------------------------------------------------------------------------

        # Create one-hot encoded arrays for entrance and exit delays
        entrance_oh_array = np.array([1 if state == entrance_delay_str else 0 for state in ALL_FOUR_DELAY_STATES])
        exit_oh_array = np.array([1 if state == exit_delay_str else 0 for state in ALL_FOUR_DELAY_STATES])

        data_for_local_df.append({
            'local_video_path': local_path,
            'entrance_labels_oh': entrance_oh_array,
            'exit_labels_oh': exit_oh_array
        })

    else:
        print(f"Warning: Video {local_path} not found in Train.csv mapping. Skipping.")

train_df_local = pd.DataFrame(data_for_local_df)

# Prepare data for train-test split for multiple outputs
X = train_df_local['local_video_path']
y_entrance = np.stack(train_df_local['entrance_labels_oh'].to_numpy())
y_exit = np.stack(train_df_local['exit_labels_oh'].to_numpy())

X_train_paths, X_val_paths, y_train_entrance_oh, y_val_entrance_oh, y_train_exit_oh, y_val_exit_oh = train_test_split(
    X, y_entrance, y_exit, test_size=0.2, random_state=42, stratify=np.argmax(y_entrance, axis=1)
)

# Combine labels into lists for the generator to handle multiple outputs
y_train = [y_train_entrance_oh, y_train_exit_oh]
y_val = [y_val_entrance_oh, y_val_exit_oh]

class VideoFrameGenerator(tf.keras.utils.Sequence):
    """
    Generates batches of preprocessed video frames for Keras models.
    Handles on-the-fly video loading, frame resizing, normalization, and sequence padding/truncation.
    """
    def __init__(
        self, video_paths, labels, sequence_length, batch_size,
        target_size=(224, 224), shuffle=True, n_classes=None, **kwargs
    ):
        super().__init__(**kwargs)
        self.video_paths = video_paths.reset_index(drop=True) # Reset index to ensure integer indexing
        self.labels = labels # Labels is now a list of two arrays: [y_entrance_oh, y_exit_oh]
        self.entrance_labels = self.labels[0]
        self.exit_labels = self.labels[1]
        self.sequence_length = sequence_length
        self.batch_size = batch_size
        self.target_size = target_size
        self.shuffle = shuffle
        # n_classes must be explicitly passed for multi-output regression
        if n_classes is None:
            raise ValueError("n_classes must be specified for VideoFrameGenerator with multi-output labels.")
        self.n_classes = n_classes # This will be 4 for each output
        self.on_epoch_end()

    def __len__(self):
        """Denotes the number of batches per epoch."""
        return math.ceil(len(self.video_paths) / self.batch_size)

    def __getitem__(self, idx):
        """Generates one batch of data."""
        # Get batch indices
        indices = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]

        # Select batch data
        batch_video_paths = self.video_paths.iloc[indices]
        batch_entrance_labels = self.entrance_labels[indices] # Select entrance labels
        batch_exit_labels = self.exit_labels[indices]     # Select exit labels

        # Initialize batch arrays for video frames and labels
        X_batch = np.zeros(
            (len(batch_video_paths), self.sequence_length, self.target_size[0], self.target_size[1], 3),
            dtype=np.float32
        )

        # y_batch now needs to hold the two 4-class one-hot encoded label arrays
        y_entrance_batch = np.zeros((len(batch_video_paths), self.n_classes), dtype=np.float32)
        y_exit_batch = np.zeros((len(batch_video_paths), self.n_classes), dtype=np.float32)

        # Process each video in the batch
        for i, video_path in enumerate(batch_video_paths):
            # Use the updated target_size and pass max_frames
            frames = load_and_preprocess_video(video_path, self.target_size, max_frames=self.sequence_length)

            # Pad or truncate frames to sequence_length
            if len(frames) < self.sequence_length:
                # Pad with zeros
                num_padding = self.sequence_length - len(frames)
                # Ensure padding frames have the correct shape
                padded_frames = frames + [np.zeros(self.target_size + (3,), dtype=np.float32)] * num_padding
                X_batch[i] = np.array(padded_frames)
            elif len(frames) > self.sequence_length:
                # Truncate
                X_batch[i] = np.array(frames[:self.sequence_length])
            else:
                X_batch[i] = np.array(frames)

            # Assign the two 4-class one-hot encoded labels
            y_entrance_batch[i] = batch_entrance_labels[i]
            y_exit_batch[i] = batch_exit_labels[i]

        # Return labels as a dictionary matching the model's named outputs
        return X_batch, {'entrance_output': y_entrance_batch, 'exit_output': y_exit_batch}

    def on_epoch_end(self):
        """Updates indices after each epoch."""
        self.indices = np.arange(len(self.video_paths))
        if self.shuffle:
            np.random.shuffle(self.indices)

# --- Instantiate generators ---
# n_classes_output is now 4 for each of the two classification outputs
n_classes_output = 4

# For small datasets, use batch_size=1 or 2 to avoid errors when the last batch is smaller
BATCH_SIZE = 1 # Keep 1 for consistency and small dataset size
SEQUENCE_LENGTH = 1200 # Modified to 1200 as per plan
TARGET_SIZE = (256, 256) # Retained from previous cell output

train_generator = VideoFrameGenerator(
    X_train_paths,
    y_train, # y_train is now a list of two arrays
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    target_size=TARGET_SIZE,
    shuffle=True,
    n_classes=n_classes_output
)

val_generator = VideoFrameGenerator(
    X_val_paths,
    y_val, # y_val is now a list of two arrays
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE, # Must be 1 since X_val has only 1 element
    target_size=TARGET_SIZE,
    shuffle=False, # Typically no shuffle for validation set
    n_classes=n_classes_output
)


## Visualizing Loaded Frames and Labels

In [12]:
# import matplotlib.pyplot as plt

# print("\n--- Visualizing First 5 Samples from Training Generator ---")
# for i in range(min(5, len(train_generator))):
#     X_batch, y_batch = train_generator[i]

#     # Assuming BATCH_SIZE is 1, so X_batch[0] is the video sequence and y_batch[0] is the label.
#     video_sequence = X_batch[0] # Shape (SEQUENCE_LENGTH, TARGET_SIZE[0], TARGET_SIZE[1], 3)
#     label = y_batch[0]          # Shape (n_classes_output,)

#     print(f"\nSample {i+1} - Label: {label}")

#     # Display the first few frames from the sequence
#     fig, axes = plt.subplots(1, 5, figsize=(20, 4))
#     fig.suptitle(f'Sample {i+1} Frames (Label: {label})', fontsize=16)

#     for j in range(min(5, len(video_sequence))): # Display first 5 frames or fewer if sequence is shorter
#         frame = video_sequence[j]
#         axes[j].imshow(frame) # Frames are already normalized to [0, 1] and RGB
#         axes[j].set_title(f'Frame {j+1}')
#         axes[j].axis('off')
#     plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent suptitle overlap
#     plt.show()

## Create the model architecture

In [13]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, TimeDistributed, Flatten, LSTM, Dense

# 2. Define the input shape for the model
# (SEQUENCE_LENGTH, TARGET_SIZE[0], TARGET_SIZE[1], 3)
input_shape = (SEQUENCE_LENGTH, TARGET_SIZE[0], TARGET_SIZE[1], 3)

# 3. Create an Input layer
input_layer = Input(shape=input_shape);

# 4. Apply TimeDistributed(Flatten()) to flatten each frame
flattened_frames = TimeDistributed(Flatten())(input_layer);

# 5. Add an LSTM layer
lstm_output = LSTM(units=128)(flattened_frames) # Using 128 units as a starting point

# 6. Add two Dense output layers with n_classes_output units and 'softmax' activation for multi-class classification
output_entrance_layer = Dense(units=n_classes_output, activation='softmax', name='entrance_output')(lstm_output)
output_exit_layer = Dense(units=n_classes_output, activation='softmax', name='exit_output')(lstm_output)

# 7. Instantiate a tf.keras.Model with two outputs
model = Model(inputs=input_layer, outputs=[output_entrance_layer, output_exit_layer])

# 8. Print the model summary
print("LSTM Model Architecture:")
model.summary()

I0000 00:00:1767981099.456008      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


LSTM Model Architecture:


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 1200, 256, │          0 │ -                 │
│ (InputLayer)        │ 256, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed    │ (None, 1200,      │          0 │ input_layer[0][0] │
│ (TimeDistributed)   │ 196608)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 128)       │ 100,729,3… │ time_distributed… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entrance_output     │ (None, 4)         │        516 │ lstm[0][0]        │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ exit_output (Dense) │ (None, 4)         │        516 │ lstm[0][0]        │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 100,730,376 (384.26 MB)

 Trainable params: 100,730,376 (384.26 MB)

 Non-trainable params: 0 (0.00 B)

## Load model

In [14]:
MODEL_PATH = '/kaggle/working/traffic_predictor.keras'

model = tf.keras.models.load_model(MODEL_PATH)

ValueError: File not found: filepath=/kaggle/working/traffic_predictor.keras. Please ensure the file is an accessible `.keras` zip file.

## Build the compiler

In [15]:
from tensorflow.keras.optimizers import Adam

# For multi-class classification with one-hot encoded labels, use categorical_crossentropy.
loss_function = ['categorical_crossentropy', 'categorical_crossentropy']

# For classification, metrics like accuracy are appropriate.
metrics_list = ['accuracy', 'accuracy'] # Changed from 'mae' to 'accuracy'

# Compile the model
model.compile(
    optimizer=Adam(),
    loss=loss_function,
    metrics=metrics_list
)

print(f"Model compiled with optimizer: Adam, loss: {loss_function}, metrics: {metrics_list}")

Model compiled with optimizer: Adam, loss: ['categorical_crossentropy', 'categorical_crossentropy'], metrics: ['accuracy', 'accuracy']


## Train the model

In [ ]:
model_checkpoint = keras.callbacks.ModelCheckpoint(
    'best_model.keras',
    monitor='loss',
    save_best_only=True,
    verbose=1
)

history = model.fit(
    train_generator,
    validation_data=val_generator,
    callbacks=model_checkpoint,
    epochs=3
)

print("Model training complete.")

Epoch 1/3


I0000 00:00:1767981161.787029     129 cuda_dnn.cc:529] Loaded cuDNN version 91002


  73/1600 ━━━━━━━━━━━━━━━━━━━━ 3:19:18 8s/step - entrance_output_accuracy: 0.6971 - entrance_output_loss: 0.9231 - exit_output_accuracy: 0.9020 - exit_output_loss: 0.2252 - loss: 1.1484

[mov,mp4,m4a,3gp,3g2,mj2 @ 0x7ad57801f2c0] moov atom not found
[18:02:16] /github/workspace/src/video/video_reader.cc:83: ERROR opening: /kaggle/working/videos/normanniles1_2025-10-20-14-40-45.mp4, Invalid data found when processing input


Error: Could not open video /kaggle/working/videos/normanniles1_2025-10-20-14-40-45.mp4 with decord: Error reading /kaggle/working/videos/normanniles1_2025-10-20-14-40-45.mp4...
 208/1600 ━━━━━━━━━━━━━━━━━━━━ 3:01:51 8s/step - entrance_output_accuracy: 0.7517 - entrance_output_loss: 0.7822 - exit_output_accuracy: 0.9461 - exit_output_loss: 0.1921 - loss: 0.9742Error: Could not open video /kaggle/working/videos/normanniles1_2025-10-22-13-17-45.mp4 with decord: Error reading /kaggle/working/videos/normanniles1_2025-10-22-13-17-45.mp4...


...

Error: Could not open video /kaggle/working/videos/normanniles1_2025-10-21-07-39-45.mp4 with decord: Error reading /kaggle/working/videos/normanniles1_2025-10-21-07-39-45.mp4...
 255/1600 ━━━━━━━━━━━━━━━━━━━━ 2:58:52 8s/step - entrance_output_accuracy: 0.8256 - entrance_output_loss: 0.5785 - exit_output_accuracy: 0.9760 - exit_output_loss: 0.1333 - loss: 0.7118

[mov,mp4,m4a,3gp,3g2,mj2 @ 0x7ad65c129040] moov atom not found
[22:49:43] /github/workspace/src/video/video_reader.cc:83: ERROR opening: /kaggle/working/videos/normanniles1_2025-10-22-13-17-45.mp4, Invalid data found when processing input


Error: Could not open video /kaggle/working/videos/normanniles1_2025-10-22-13-17-45.mp4 with decord: Error reading /kaggle/working/videos/normanniles1_2025-10-22-13-17-45.mp4...
 401/1600 ━━━━━━━━━━━━━━━━━━━━ 2:39:07 8s/step - entrance_output_accuracy: 0.8163 - entrance_output_loss: 0.5997 - exit_output_accuracy: 0.9753 - exit_output_loss: 0.1361 - loss: 0.7358

[mov,mp4,m4a,3gp,3g2,mj2 @ 0x7ad6f00429c0] moov atom not found
[23:09:01] /github/workspace/src/video/video_reader.cc:83: ERROR opening: /kaggle/working/videos/normanniles1_2025-10-20-11-44-45.mp4, Invalid data found when processing input


Error: Could not open video /kaggle/working/videos/normanniles1_2025-10-20-11-44-45.mp4 with decord: Error reading /kaggle/working/videos/normanniles1_2025-10-20-11-44-45.mp4...
 434/1600 ━━━━━━━━━━━━━━━━━━━━ 2:34:28 8s/step - entrance_output_accuracy: 0.8153 - entrance_output_loss: 0.6021 - exit_output_accuracy: 0.9752 - exit_output_loss: 0.1364 - loss: 0.7385

In [ ]:
final_model_path = '/kaggle/working/traffic_predictor.keras'
model.save(final_model_path)

## Visualize the training result

In [ ]:
import matplotlib.pyplot as plt

# Extract data from history object
hist = history.history
train_loss_entrance = hist['entrance_output_loss']
train_loss_exit = hist['exit_output_loss']
val_loss_entrance = hist['val_entrance_output_loss']
val_loss_exit = hist['val_exit_output_loss']

train_accuracy_entrance = hist['entrance_output_accuracy']
train_accuracy_exit = hist['exit_output_accuracy']
val_accuracy_entrance = hist['val_entrance_output_accuracy']
val_accuracy_exit = hist['val_exit_output_accuracy']

epochs = range(1, len(train_loss_entrance) + 1)

# Plot training and validation loss for entrance
plt.figure(figsize=(18, 6))

plt.subplot(1, 4, 1) # 1 row, 4 columns, first plot
plt.plot(epochs, train_loss_entrance, 'b', label='Training Entrance Loss')
plt.plot(epochs, val_loss_entrance, 'r', label='Validation Entrance Loss')
plt.title('Training and Validation Entrance Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Plot training and validation accuracy for entrance
plt.subplot(1, 4, 2) # 1 row, 4 columns, second plot
plt.plot(epochs, train_accuracy_entrance, 'b', label='Training Entrance Accuracy')
plt.plot(epochs, val_accuracy_entrance, 'r', label='Validation Entrance Accuracy')
plt.title('Training and Validation Entrance Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Plot training and validation loss for exit
plt.subplot(1, 4, 3) # 1 row, 4 columns, third plot
plt.plot(epochs, train_loss_exit, 'b', label='Training Exit Loss')
plt.plot(epochs, val_loss_exit, 'r', label='Validation Exit Loss')
plt.title('Training and Validation Exit Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Plot training and validation accuracy for exit
plt.subplot(1, 4, 4) # 1 row, 4 columns, fourth plot
plt.plot(epochs, train_accuracy_exit, 'b', label='Training Exit Accuracy')
plt.plot(epochs, val_accuracy_exit, 'r', label='Validation Exit Accuracy')
plt.title('Training and Validation Exit Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

# Task
Modify cell `6b29a501` to add a boolean variable `DECODORD_USE_GPU` at the top and update the `load_and_preprocess_video` function to switch `decord.VideoReader` context between `cpu(0)` and `gpu(0)` based on this variable.

## Add Decord Device Switch Variable

### Subtask:
Modify cell `6b29a501` to include a boolean variable `DECODORD_USE_GPU` at the top. This variable will determine the context (`ctx=cpu(0)` or `ctx=gpu(0)`) used by `decord.VideoReader` within the `load_and_preprocess_video` function. This makes the resource allocation easily configurable.


## Summary:

### Data Analysis Key Findings
- The primary objective of the current step is a code modification: to add a boolean variable `DECODORD_USE_GPU` at the top of cell `6b29a501`.
- This variable will control the device context (`cpu(0)` or `gpu(0)`) for `decord.VideoReader` within the `load_and_preprocess_video` function, making resource allocation easily configurable.

### Insights or Next Steps
- The immediate next step is to implement the specified code modification in cell `6b29a501`.
- This modification aims to enhance the flexibility and configurability of video processing by allowing easy switching between CPU and GPU resources based on the `DECODORD_USE_GPU` variable.
